In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from geopy.geocoders import ArcGIS
import time
import pandas as pd

# 1. Khởi tạo
geolocator = ArcGIS(user_agent="solar_data_explorer_v3")

# 2. Lấy tọa độ duy nhất
unique_coords = site_details[['lat', 'Lon']].drop_duplicates().reset_index(drop=True)
location_map = {}

print(f"Đang tra cứu tên địa danh cho {len(unique_coords)} cụm tọa độ...")

for index, row in unique_coords.iterrows():
    lat, lon = row['lat'], row['Lon']
    try:
        location = geolocator.reverse((lat, lon), timeout=10)
        location_map[(lat, lon)] = location.address if location else "Không tìm thấy"
        print(f"Tọa độ {index+1} xong.")
    except:
        location_map[(lat, lon)] = "Lỗi kết nối"
    time.sleep(0.5)

# 3. Ánh xạ địa chỉ vào TOÀN BỘ 42 trạm
site_details_final = site_details.copy()
site_details_final['Address'] = site_details_final.apply(lambda r: location_map.get((r['lat'], r['Lon']), "N/A"), axis=1)

# Hiển thị toàn bộ danh sách 42 dòng
print("\nDanh sách đầy đủ 42 trạm:")
pd.set_option('display.max_rows', None) # Hiển thị hết các dòng
display(site_details_final[['SiteKey', 'lat', 'Lon', 'Address']])

# Lưu lại file
site_details_final.to_csv('Solar_Sites_Full_Addresses.csv', index=False)
print(f"\nĐã lưu file đầy đủ 42 trạm vào: Solar_Sites_Full_Addresses.csv")

In [ ]:
# Kiểm tra nhanh tính nhất quán của dữ liệu trước khi crawl
print("Kiểm tra 5 trạm đầu tiên:")
display(site_details_final[['SiteKey', 'lat', 'Lon', 'Address']].head())

# Kiểm tra xem có trạm nào bị thiếu tọa độ không
missing_coords = site_details_final[['lat', 'Lon']].isnull().any(axis=1).sum()
print(f"\nSố lượng trạm thiếu tọa độ: {missing_coords}")

In [ ]:
import requests
import pandas as pd
import time
from google.colab import files

# Sử dụng bảng dữ liệu đã có địa chỉ từ bước tra cứu ArcGIS
data_source = site_details_final

url = "https://power.larc.nasa.gov/api/temporal/daily/point"
parameters = "ALLSKY_SFC_SW_DWN,CLRSKY_SFC_SW_DWN,PRECTOTCORR,T2M"

all_regional_data = []

print(f"Bắt đầu thu thập dữ liệu NASA cho {len(data_source)} trạm kèm địa chỉ...")

for index, row in data_source.iterrows():
    site_key = row['SiteKey']
    lat = row['lat']
    lon = row['Lon']
    address = row['Address']

    print(f"[{index+1}/{len(data_source)}] Đang lấy dữ liệu Site {site_key}: {address[:50]}...")

    query_params = {
        "start": 20200101,
        "end": 20220430,
        "latitude": lat,
        "longitude": lon,
        "community": "re",
        "parameters": parameters,
        "format": "json",
        "units": "metric",
        "time-standard": "lst"
    }

    try:
        response = requests.get(url, params=query_params, timeout=30)
        if response.status_code == 200:
            data = response.json()["properties"]["parameter"]
            df_temp = pd.DataFrame(data)
            df_temp.index.name = "Date"
            df_temp.reset_index(inplace=True)

            # Chèn thông tin định danh và địa chỉ
            df_temp["SiteKey"] = site_key
            df_temp["Address"] = address
            df_temp["Latitude"] = lat
            df_temp["Longitude"] = lon

            all_regional_data.append(df_temp)
        else:
            print(f" ! Lỗi HTTP {response.status_code} tại Site {site_key}")
    except Exception as e:
        print(f" ! Lỗi kết nối tại Site {site_key}: {e}")

    # Delay để tránh rate limit
    time.sleep(1)

if all_regional_data:
    df_final = pd.concat(all_regional_data, ignore_index=True)
    df_final["Date"] = pd.to_datetime(df_final["Date"], format="%Y%m%d")

    # Sắp xếp các cột: Date, SiteKey, Address lên trước
    cols_order = ['Date', 'SiteKey', 'Address', 'Latitude', 'Longitude']
    other_cols = [c for c in df_final.columns if c not in cols_order]
    df_final = df_final[cols_order + other_cols]

    output_name = "NASA_Solar_Weather_Data_With_Addresses.csv"
    df_final.to_csv(output_name, index=False)

    print(f"\n--- HOÀN THÀNH: Đã lưu {len(df_final)} bản ghi ---")
    display(df_final.head())
    files.download(output_name)
else:
    print("Không thu thập được dữ liệu.")

In [ ]:
from geopy.geocoders import ArcGIS
import pandas as pd
import time

geolocator = ArcGIS(user_agent="vn_solar_provinces_all")

# Danh sách 63 tỉnh thành của Việt Nam
vn_provinces = [
    "An Giang", "Bà Rịa – Vũng Tàu", "Bắc Giang", "Bắc Kạn", "Bạc Liêu",
    "Bắc Ninh", "Bến Tre", "Bình Định", "Bình Dương", "Bình Phước",
    "Bình Thuận", "Cà Mau", "Cao Bằng", "Cần Thơ", "Đà Nẵng",
    "Đắk Lắk", "Đắk Nông", "Điện Biên", "Đồng Nai", "Đồng Tháp",
    "Gia Lai", "Hà Giang", "Hà Nam", "Hà Nội", "Hà Tĩnh",
    "Hải Dương", "Hải Phòng", "Hậu Giang", "Hòa Bình", "Thành phố Hồ Chí Minh",
    "Hưng Yên", "Khánh Hòa", "Kiên Giang", "Kon Tum", "Lai Châu",
    "Lâm Đồng", "Lạng Sơn", "Lào Cai", "Long An", "Nam Định",
    "Nghệ An", "Ninh Bình", "Ninh Thuận", "Phú Thọ", "Phú Yên",
    "Quảng Bình", "Quảng Nam", "Quảng Ngãi", "Quảng Ninh", "Quảng Trị",
    "Sóc Trăng", "Sơn La", "Tây Ninh", "Thái Bình", "Thái Nguyên",
    "Thanh Hóa", "Thừa Thiên Huế", "Tiền Giang", "Trà Vinh", "Tuyên Quang",
    "Vĩnh Long", "Vĩnh Phúc", "Yên Bái"
]

vn_data_all_provinces = []
print(f"Đang tìm tọa độ cho nhà máy solar tại {len(vn_provinces)} tỉnh/thành...")

for i, province in enumerate(vn_provinces):
    search_query = f"Điện mặt trời {province}"
    try:
        location = geolocator.geocode(search_query, timeout=10)
        if location:
            vn_data_all_provinces.append({
                'SiteKey': f"VN_PROVINCE_{i+1}",
                'lat': location.latitude,
                'Lon': location.longitude,
                'Address': location.address,
                'OriginalName': province,
                'FoundPlant': search_query # Store the actual search query that returned a result
            })
            print(f"✅ Đã tìm thấy cho {province}: {location.address}")
        else:
            vn_data_all_provinces.append({
                'SiteKey': f"VN_PROVINCE_{i+1}",
                'lat': None,
                'Lon': None,
                'Address': "Không tìm thấy",
                'OriginalName': province,
                'FoundPlant': None
            })
            print(f"❌ Không tìm thấy điện mặt trời tại {province}")
    except Exception as e:
        vn_data_all_provinces.append({
            'SiteKey': f"VN_PROVINCE_{i+1}",
            'lat': None,
            'Lon': None,
            'Address': "Lỗi kết nối",
            'OriginalName': province,
            'FoundPlant': None
        })
        print(f"⚠️ Lỗi khi tra cứu '{search_query}': {e}")
    time.sleep(0.5) # Delay để tránh rate limit

df_vn_all_provinces = pd.DataFrame(vn_data_all_provinces)
print(f"\nHoàn thành! Tìm được {len(df_vn_all_provinces[df_vn_all_provinces['lat'].notna()])} trạm có tọa độ trong tổng số {len(vn_provinces)} tỉnh/thành.")
display(df_vn_all_provinces)

# Gán vào site_details_final để chuẩn bị crawl NASA
site_details_final = df_vn_all_provinces

In [ ]:
import requests
import pandas as pd
import time
from google.colab import files

# Sử dụng danh sách nhà máy Việt Nam vừa tìm được (đã được cập nhật từ bước 63 tỉnh)
data_source = site_details_final

url = "https://power.larc.nasa.gov/api/temporal/daily/point"
parameters = "ALLSKY_SFC_SW_DWN,CLRSKY_SFC_SW_DWN,PRECTOTCORR,T2M"

all_vn_data = []

# Lọc bỏ các hàng không có tọa độ trước khi crawl NASA
data_source_valid_coords = data_source.dropna(subset=['lat', 'Lon']).copy()

print(f"Bắt đầu crawl dữ liệu NASA cho {len(data_source_valid_coords)} nhà máy/địa điểm tại Việt Nam có tọa độ...")

if data_source_valid_coords.empty:
    print("Không có địa điểm nào có tọa độ hợp lệ để crawl dữ liệu NASA.")
else:
    for index, row in data_source_valid_coords.iterrows():
        site_key = row['SiteKey']
        lat = row['lat']
        lon = row['Lon']
        name = row['OriginalName']

        print(f"[{index+1}/{len(data_source_valid_coords)}] Đang lấy dữ liệu: {name}...")

        query_params = {
            "start": 20210101, # Lấy dữ liệu từ 2021
            "end": 20231231,   # Đến hết 2023
            "latitude": lat,
            "longitude": lon,
            "community": "re",
            "parameters": parameters,
            "format": "json",
            "units": "metric",
            "time-standard": "lst"
        }

        try:
            response = requests.get(url, params=query_params, timeout=30)
            if response.status_code == 200:
                data = response.json()["properties"]["parameter"]
                df_temp = pd.DataFrame(data)
                df_temp.index.name = "Date"
                df_temp.reset_index(inplace=True)

                # Chèn thông tin định danh
                df_temp["SiteKey"] = site_key
                df_temp["PlantName"] = name
                df_temp["Latitude"] = lat
                df_temp["Longitude"] = lon

                all_vn_data.append(df_temp)
            else:
                print(f" ! Lỗi HTTP {response.status_code} tại {name}")
        except Exception as e:
            print(f" ! Lỗi kết nối tại {name}: {e}")

        # Nghỉ 1s giữa các request để tránh bị khóa IP
        time.sleep(1)

    if all_vn_data:
        df_vn_weather = pd.concat(all_vn_data, ignore_index=True)
        df_vn_weather["Date"] = pd.to_datetime(df_vn_weather["Date"], format="%Y%m%d")

        # Lưu kết quả
        output_file = "VN_Solar_NASA_Data_All_Provinces.csv"
        df_vn_weather.to_csv(output_file, index=False)

        print(f"\n--- HOÀN THÀNH: Đã tải {len(df_vn_weather)} dòng dữ liệu ---\nKết quả đã lưu vào: {output_file}")
        display(df_vn_weather.head())
        files.download(output_file)
    else:
        print("Không lấy được dữ liệu nào từ NASA.")

In [ ]:
import pandas as pd
from google.colab import files

# 1. Lưu file thông tin tọa độ/địa chỉ của 63 tỉnh (Site Details)
if 'site_details_final' in globals():
    site_details_file = 'VN_Solar_Site_Details.csv'
    site_details_final.to_csv(site_details_file, index=False)
    print(f"✅ Đã chuẩn bị file chi tiết tọa độ: {site_details_file}")
else:
    print("⚠️ Không tìm thấy biến site_details_final. Vui lòng chạy lại ô quét tọa độ.")

# 2. Danh sách các file cần tải về máy
# Bao gồm cả file site details vừa tạo và file NASA data tổng hợp
files_to_download = [
    'VN_Solar_Site_Details.csv',
    'VN_Solar_NASA_Data_All_Provinces.csv'
]

print("\n--- Bắt đầu tải file ---")
for file_name in files_to_download:
    try:
        # Kiểm tra file có tồn tại trong hệ thống không trước khi tải
        import os
        if os.path.exists(file_name):
            print(f"Đang tải: {file_name}...")
            files.download(file_name)
        else:
            print(f"❌ Không tìm thấy file {file_name}. Hãy chắc chắn bạn đã chạy xong bước Crawl dữ liệu.")
    except Exception as e:
        print(f"❌ Lỗi khi tải {file_name}: {e}")

In [ ]:
import pandas as pd
from google.colab import files

# 1. Lưu danh sách thông tin các nhà máy Việt Nam (SiteKey, Tọa độ, Địa chỉ)
if 'df_vn_final' in globals():
    vn_sites_output = "VN_Solar_Site_Details.csv"
    df_vn_final.to_csv(vn_sites_output, index=False)
    print(f"✅ Đã tạo file danh sách nhà máy VN: {vn_sites_output}")

    # 2. Tải file về máy
    files.download(vn_sites_output)
else:
    print("❌ Không tìm thấy dữ liệu df_vn_final trong bộ nhớ.")